# Fase 2 — Notebook 2: limpieza, depuración y transformación

**Proyecto:** Sobreduración en la titulación de la educación superior chilena (2020–2025)
**Equipo:** «Integrante 1», «Integrante 2», «Integrante 3»  ·  **Grupo:** «N»

---

**Objetivo de este notebook (OE4 y OE5).** Ejecutar el pipeline de depuración
sobre el conjunto consolidado y derivar las variables analíticas del proyecto,
dejando registrada cada decisión técnica y su efecto sobre el volumen de datos.

Cada paso sigue la misma estructura: **qué problema resuelve**, **qué
alternativa se descartó y por qué**, y **cuántos registros afecta**. El registro
acumulado se materializa en un objeto `BitacoraLimpieza`, que al final del
notebook se exporta como tabla de trazabilidad.

## 1. Entorno y carga del consolidado

In [ ]:
# Celda de arranque: hace importable el paquete `src` sin instalar el proyecto
# y funciona igual si el notebook se abre desde la raiz o desde su subcarpeta.
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

from src import config

print("Raiz del proyecto:", RAIZ)
for clave, valor in config.describir_entorno().items():
    print(f"  {clave:12s} {valor}")

In [ ]:
from src import ingesta, limpieza, transformacion, viz

viz.aplicar_estilo()

df = ingesta.cargar_consolidado()
bitacora = limpieza.BitacoraLimpieza()
bitacora.registrar("consolidado_inicial", "Unión de los seis años", len(df), len(df))

print(f"Punto de partida: {len(df):,} registros x {df.shape[1]} columnas")

## 2. Paso 1 — Códigos centinela a valores nulos explícitos

**Problema.** El SIES no usa celdas vacías: codifica la ausencia de información
con valores numéricos válidos (9995, 9998, 9999 y 1900 en el año de ingreso;
19000101 en las fechas; 0 en el semestre). Si se procesan como números, un
estudiante con año de ingreso 9999 arrojaría una duración negativa de miles de
semestres.

**Alternativa descartada.** Filtrar directamente esas filas en este punto.
Se descartó porque impediría cuantificar cuánta información falta y en qué
variables; el diagnóstico debe preceder al tratamiento.

**Decisión.** Convertir los centinelas en `NA` mediante tipos enteros anulables
(`Int32`, `Int8`), sin eliminar ninguna fila todavía.

In [ ]:
antes_nulos = int(df.isna().sum().sum())
df = limpieza.marcar_sentinelas(df, bitacora)
despues_nulos = int(df.isna().sum().sum())

print(f"Celdas nulas antes:   {antes_nulos:,}")
print(f"Celdas nulas después: {despues_nulos:,}")
print(f"Ausencias hechas explícitas: {despues_nulos - antes_nulos:,}\n")

reporte = limpieza.reporte_faltantes(df)
print(reporte.loc[reporte["nulos"] > 0].to_string(index=False))

## 3. Paso 2 — Normalización de las categorías de texto

**Problema.** Las columnas de texto provienen de sistemas institucionales
distintos y llegan con capitalización y espaciado heterogéneos. Al agrupar, dos
escrituras de la misma categoría se contarían por separado.

**Decisión.** Homogeneizar con `str.strip()`, colapso de espacios múltiples y
capitalización tipo título, y convertir a `category` para reducir memoria.

In [ ]:
ejemplo_antes = df["nomb_inst"].astype("string").head(3).tolist()

df = limpieza.normalizar_categorias(df, bitacora=bitacora)

print("Antes:", ejemplo_antes)
print("Después:", df["nomb_inst"].astype("string").head(3).tolist())
print(f"\nCategorías únicas en region_sede: {df['region_sede'].nunique()}")
print(f"Categorías únicas en nomb_carrera: {df['nomb_carrera'].nunique():,}")

## 4. Paso 3 — Restricción al nivel de pregrado

**Problema.** El conjunto mezcla pregrado, posgrado y postítulo. Un magíster de
4 semestres y una carrera de medicina de 14 no son comparables en términos de
sobreduración: el indicador perdería sentido si se promediaran juntos.

**Alternativa descartada.** Analizar los tres niveles y controlar por nivel en
el análisis. Se descartó por alcance: la Fase 2 debe entregar un conjunto
analítico único y homogéneo, y el pregrado concentra el 76,7 % de los registros.

**Decisión.** Filtrar `nivel_global == "Pregrado"`, dejando documentado el
volumen excluido.

In [ ]:
composicion_nivel = df["nivel_global"].value_counts()
print(composicion_nivel.to_string())
print()

df = limpieza.filtrar_nivel(df, bitacora=bitacora)
print(f"Registros de pregrado retenidos: {len(df):,}")

## 5. Paso 4 — Eliminación de duplicados

**Problema.** Un mismo título podría aparecer repetido por reprocesos
administrativos.

**Precaución 1.** Un estudiante con **dos títulos distintos** en el mismo año no
es un duplicado: es un hecho real del sistema (por ejemplo, técnico de nivel
superior y luego continuidad profesional). La clave de deduplicación incluye por
eso la carrera y la fecha.

**Precaución 2.** Las filas sin `mrun` quedan **fuera del cotejo**: como pandas
considera iguales dos valores nulos, incluirlas colapsaría estudiantes distintos
en un solo registro.

In [ ]:
antes = len(df)
df = limpieza.eliminar_duplicados(df, bitacora=bitacora)

print(f"Duplicados eliminados: {antes - len(df):,}")
print(f"Registros restantes: {len(df):,}")

## 6. Paso 5 — Descarte de registros sin insumos para el cálculo

**Problema.** Sin año de ingreso, sin semestre de ingreso o sin fecha de
titulación es imposible reconstruir la duración real.

**Alternativa descartada.** Imputar el año de ingreso con la mediana del
programa. Se descartó por una razón de fondo: imputar el inicio de la trayectoria
equivale a **inventar la variable que el proyecto busca medir**, y contaminaría
el resultado principal con un artefacto del método.

**Decisión.** Descartar esos registros y declarar explícitamente el volumen
perdido para acotar el sesgo. Los códigos 9998 y 9999 identifican a estudiantes
provenientes de otro programa o institución, un subgrupo cuyo tiempo total de
estudios no es observable en esta base.

In [ ]:
antes = len(df)
faltan_ingreso = int(df["anio_ing_carr_ori"].isna().sum())

df = limpieza.descartar_sin_insumos(df, bitacora=bitacora)

print(f"Registros sin año de ingreso utilizable: {faltan_ingreso:,}")
print(f"Descartados en total: {antes - len(df):,} ({100*(antes-len(df))/antes:.2f}%)")
print(f"Registros restantes: {len(df):,}")

## 7. Transformación: construcción de las variables analíticas

Con el conjunto depurado se derivan las variables que responden a las preguntas
del proyecto. La fórmula central es:

$$\text{duración real} = (\text{año título} - \text{año ingreso}) \times 2
+ (\text{semestre título} - \text{semestre ingreso}) + 1$$

El `+1` es deliberado: hace que la duración se mida en **semestres cursados** y
no en semestres transcurridos, de modo que ingresar y titularse dentro del mismo
semestre registre 1 y no 0.

El semestre de titulación no viene en la base y se imputa desde el mes: hasta
julio, primer semestre; desde agosto, segundo (`config.MES_CORTE_PRIMER_SEMESTRE`).

A partir de ella se construyen:

| Variable | Definición | Uso |
|----------|-----------|-----|
| `duracion_real_sem` | Semestres efectivamente cursados | Base del análisis |
| `sobreduracion_sem` | Duración real − duración teórica | **Variable objetivo** |
| `indice_duracion` | Duración real ÷ duración teórica | Comparación entre planes de distinto largo |
| `titulacion_oportuna` | Verdadero si la sobreduración es ≤ 0 | Indicador binario para F3 |
| `categoria_rezago` | Oportuna / leve (≤2) / moderado (≤4) / severo (>4) | Segmentación |
| `edad_titulacion` | Edad cumplida al titularse | Caracterización del perfil |
| `genero` | Etiqueta legible de `gen_alu` | Análisis de brechas |

### 7.1 Verificación previa sobre un caso conocido

Antes de aplicar la fórmula a más de un millón de registros se comprueba sobre
tres casos construidos a mano, incluyendo un caso límite (ingreso y titulación
en el mismo semestre).

In [ ]:
casos = pd.DataFrame([
    {"caso": "Trayectoria de 7 años en plan de 10 semestres",
     "anio_ing_carr_ori": 2018, "sem_ing_carr_ori": 1,
     "fecha_obtencion_titulo": 20241220, "dur_total_carr": 10,
     "cat_periodo": 2024, "mrun": 1, "gen_alu": 2, "fec_nac_alu": 199805,
     "nivel_global": "Pregrado"},
    {"caso": "Titulación en el plazo teórico exacto",
     "anio_ing_carr_ori": 2020, "sem_ing_carr_ori": 1,
     "fecha_obtencion_titulo": 20241220, "dur_total_carr": 10,
     "cat_periodo": 2024, "mrun": 2, "gen_alu": 1, "fec_nac_alu": 200003,
     "nivel_global": "Pregrado"},
    {"caso": "Caso límite: ingreso y título en el mismo semestre",
     "anio_ing_carr_ori": 2024, "sem_ing_carr_ori": 1,
     "fecha_obtencion_titulo": 20240310, "dur_total_carr": 4,
     "cat_periodo": 2024, "mrun": 3, "gen_alu": 2, "fec_nac_alu": 199001,
     "nivel_global": "Pregrado"},
])

verificacion = transformacion.construir_dataset_analitico(casos)
print(verificacion[["caso", "duracion_real_sem", "dur_total_carr",
                    "sobreduracion_sem", "indice_duracion",
                    "categoria_rezago", "edad_titulacion"]].to_string(index=False))

assert verificacion.loc[0, "duracion_real_sem"] == 14
assert verificacion.loc[1, "sobreduracion_sem"] == 0
assert verificacion.loc[2, "duracion_real_sem"] == 1
print("\nLas tres verificaciones manuales coinciden con la implementación.")

### 7.2 Aplicación al conjunto completo

In [ ]:
df = transformacion.construir_dataset_analitico(df, bitacora)

derivadas = ["duracion_real_sem", "dur_total_carr", "sobreduracion_sem",
             "indice_duracion", "edad_titulacion"]
print(df[derivadas].describe().round(2).to_string())
print(f"\nColumnas tras la transformación: {df.shape[1]}")

## 8. Tratamiento de valores atípicos

**Problema.** La reconstrucción puede producir valores implausibles cuando el
año de ingreso registrado es incorrecto: duraciones negativas o trayectorias de
más de 20 años.

**Decisión.** Marcar y excluir los registros fuera de los rangos declarados en
`config.py`: duración real entre 1 y 40 semestres, edad de titulación entre 15 y
90 años, y duración teórica mayor que cero. Los umbrales son **explícitos y
parametrizables**, no constantes escondidas en el código.

Un registro sin fecha de nacimiento también se marca como atípico: su edad no
puede validarse, y dejar el indicador en nulo haría que el filtro descartara
filas de manera silenciosa.

In [ ]:
resumen_atipicos = pd.DataFrame([
    {"criterio": "Duración real fuera de [1, 40] semestres",
     "registros": int(df["atipico_duracion"].sum())},
    {"criterio": "Edad de titulación fuera de [15, 90] años o no verificable",
     "registros": int(df["atipico_edad"].sum())},
    {"criterio": "Duración teórica del plan igual a cero",
     "registros": int(df["atipico_teorica"].sum())},
    {"criterio": "Total de registros marcados (unión)",
     "registros": int(df["es_atipico"].sum())},
])
resumen_atipicos["porcentaje"] = (100 * resumen_atipicos["registros"] / len(df)).round(3)
print(resumen_atipicos.to_string(index=False))

print("\nEjemplos de registros atípicos por duración:")
columnas_muestra = ["cat_periodo", "anio_ing_carr_ori", "fecha_obtencion_titulo",
                    "duracion_real_sem", "dur_total_carr", "nomb_carrera"]
print(df.loc[df["atipico_duracion"], columnas_muestra].head(5).to_string(index=False))

In [ ]:
df = transformacion.filtrar_atipicos(df, bitacora)
print(f"Dataset analítico final: {len(df):,} registros x {df.shape[1]} columnas")

## 9. Bitácora de limpieza: trazabilidad completa

Esta tabla es la evidencia central de la Fase 2: permite reconstruir, registro a
registro, cómo se pasó del archivo original al conjunto analítico.

In [ ]:
tabla_bitacora = bitacora.a_dataframe()
print(tabla_bitacora.to_string(index=False))

inicial = tabla_bitacora.loc[0, "filas_antes"]
final = tabla_bitacora.iloc[-1]["filas_despues"]
print(f"\nRetención global: {final:,} de {inicial:,} registros "
      f"({100*final/inicial:.1f}%)")

tabla_bitacora.to_csv(config.TABLES_DIR / "bitacora_limpieza.csv", index=False)
print(f"Bitácora exportada a {config.TABLES_DIR / 'bitacora_limpieza.csv'}")

## 10. Distribución de las variables construidas

In [ ]:
fig = viz.histograma(
    df["sobreduracion_sem"],
    "Distribución de la sobreduración en pregrado (2020-2025)",
    "Semestres por sobre la duración teórica del plan",
    bins=50,
    referencia=0,
    nombre_archivo="f2_2_distribucion_sobreduracion",
)

print(f"Media:    {df['sobreduracion_sem'].mean():.2f} semestres")
print(f"Mediana:  {df['sobreduracion_sem'].median():.0f} semestres")
print(f"Titulación oportuna: {100*df['titulacion_oportuna'].mean():.1f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
bins = range(0, 26)
ax.hist(df["dur_total_carr"], bins=bins, alpha=0.65,
        color=config.PALETA["primario"], label="Duración teórica del plan")
ax.hist(df["duracion_real_sem"], bins=bins, alpha=0.65,
        color=config.PALETA["secundario"], label="Duración real observada")
ax.set_xlabel("Semestres")
ax.set_ylabel("Titulados")
ax.set_title("Duración teórica frente a duración real en pregrado (2020-2025)")
ax.legend()
fig.tight_layout()
viz.guardar(fig, "f2_2_teorica_vs_real")

print(f"Duración teórica mediana: {df['dur_total_carr'].median():.0f} semestres")
print(f"Duración real mediana:    {df['duracion_real_sem'].median():.0f} semestres")
print(f"Desplazamiento de la distribución: "
      f"{df['duracion_real_sem'].mean() - df['dur_total_carr'].mean():.2f} semestres en promedio")

In [ ]:
distribucion_rezago = (
    100 * df["categoria_rezago"].value_counts(normalize=True)
).round(1).reindex(["Oportuna", "Rezago leve", "Rezago moderado", "Rezago severo"])

fig = viz.barras(
    distribucion_rezago,
    "Distribución de titulados según categoría de rezago",
    "Porcentaje de titulados",
    color=config.PALETA["secundario"],
    nombre_archivo="f2_2_categorias_rezago",
)
print(distribucion_rezago.to_string())

## 11. Persistencia del conjunto analítico

El resultado se guarda en Parquet (formato de trabajo, tipado y comprimido) y
adicionalmente como muestra aleatoria en CSV, que sí puede versionarse en GitHub
y permite inspeccionar el resultado sin ejecutar el pipeline completo.

In [ ]:
df.to_parquet(config.PARQUET_LIMPIO, index=False)
muestra = df.sample(5_000, random_state=42)
muestra.to_csv(config.MUESTRA_CSV, index=False)

print(f"Dataset analítico: {config.PARQUET_LIMPIO.name} "
      f"({config.PARQUET_LIMPIO.stat().st_size/1024**2:,.1f} MB)")
print(f"Muestra versionable: {config.MUESTRA_CSV.name} "
      f"({config.MUESTRA_CSV.stat().st_size/1024**2:,.1f} MB, {len(muestra):,} filas)")

print("\nEsquema final del conjunto analítico:")
esquema = pd.DataFrame({
    "columna": df.columns,
    "tipo": [str(t) for t in df.dtypes],
    "nulos": df.isna().sum().to_numpy(),
})
print(esquema.to_string(index=False))

## 12. Cierre del notebook

Se pasó de **1.710.167 registros crudos** a un conjunto analítico de
**1.246.760 registros de pregrado** con siete variables derivadas, aplicando
seis pasos de depuración documentados uno a uno.

La pérdida acumulada se concentra en dos decisiones deliberadas y justificadas:
la exclusión de posgrado y postítulo (no comparables) y el descarte de registros
sin año de ingreso utilizable (no reconstruibles).

> **Continuar en:** `F2/F2_3_Validacion_Analisis.ipynb`